# NB06 · Semana 14 — Serving: gold en PostgreSQL + Metabase

**Ingeniería de Datos — LCD · UCA Rosario · 2026 · grupo07**  
Versión: **resuelta para el proyecto URLhaus** (`fct_amenazas_mensual`)

## Objetivo

Cerrar el pipeline: **materializar la capa gold** en PostgreSQL (warehouse de serving) y construir un **tablero en Metabase** para usuarios de negocio.

> Base de la **Entrega 4 (E4)**.

### Cómo encaja con lo que ya hace Airflow

```
URLhaus API ─▶ bronze (Delta, MinIO) ─▶ dbt: stg_urlhaus (silver) ─▶ fct_amenazas_mensual (gold, DuckDB)
                                                                              │
                                     DAG pipeline_grupo07 · tarea publicar_gold (todos los días)
                                                                              ▼
                                     PostgreSQL · base warehouse · grupo07.fct_amenazas_mensual ─▶ Metabase
```

El DAG **ya publica** la gold en PostgreSQL todos los días. Este notebook:

1. Lee la gold de DuckDB (la que generó NB04 en el JupyterHub) y la valida.
2. La publica en **la misma tabla y con la misma lógica que el DAG**, sin pisar datos más nuevos.
3. Crea **features reproducibles** (una vista SQL) y documenta el dataset dentro de PostgreSQL (governance).
4. Deja la guía para armar el **tablero en Metabase**.

In [1]:
# === Configuración del entorno — NO tocar ===
import os, sys
_d = os.getcwd()
while True:
    if os.path.exists(os.path.join(_d, "entorno.py")):
        sys.path.insert(0, _d); break
    _p = os.path.dirname(_d)
    if _p == _d:
        raise FileNotFoundError("No encuentro entorno.py subiendo desde el directorio actual.")
    _d = _p
from entorno import BASE, info, pg_url
try:
    from entorno import DBT_DIR
except ImportError:
    DBT_DIR = os.path.join(BASE, "dbt")
info()
print("dbt en:", DBT_DIR)

Entorno detectado: servidor de la cátedra  (grupo: grupo07)
  Lake / bucket   : grupo07   →   bronze en s3://grupo07/bronze/series
  PostgreSQL      : base 'warehouse', usuario 'grupo07', host 'postgres'
  Material en     : /home/jovyan/work
dbt en: /home/jovyan/work/dbt


## 1. Leer la gold desde DuckDB

La capa gold fue calculada por dbt (NB04) en `work/dbt/warehouse.duckdb`. La leemos para publicarla.

> Si da error de *lock*, cerrá (o reiniciá) el kernel de NB04: DuckDB no deja abrir el archivo mientras otro proceso lo tiene abierto para escribir.

In [2]:
import duckdb
ruta_duckdb = os.path.join(DBT_DIR, "warehouse.duckdb")
if not os.path.exists(ruta_duckdb):
    raise FileNotFoundError(f"No existe {ruta_duckdb}: corré `dbt build` (NB04) primero.")

con = duckdb.connect(ruta_duckdb, read_only=True)
gold = con.sql("""
    select cast(anio as integer)              as anio,
           cast(mes as integer)               as mes,
           tipo_amenaza,
           cast(cantidad_amenazas as bigint)  as cantidad_amenazas
    from fct_amenazas_mensual
    order by anio, mes, tipo_amenaza
""").df()
con.close()

# Mismos controles que los tests de dbt + la clave del grano (anio, mes, tipo_amenaza)
assert len(gold) > 0, "La gold está vacía"
assert gold[["anio", "mes", "tipo_amenaza"]].notna().all().all(), "Hay nulos en la clave"
assert gold["mes"].between(1, 12).all(), "Hay meses fuera de 1..12"
assert not gold.duplicated(["anio", "mes", "tipo_amenaza"]).any(), "Grano duplicado"

print(gold.shape, "·", int(gold.cantidad_amenazas.sum()), "amenazas")
print(f"Desde {gold.anio.iloc[0]}-{gold.mes.iloc[0]:02d} hasta {gold.anio.iloc[-1]}-{gold.mes.iloc[-1]:02d}")
gold.tail()

(90, 4) · 75519 amenazas
Desde 2018-06 hasta 2026-10


,anio,mes,tipo_amenaza,cantidad_amenazas
85,2026,6,malware_download,4734
86,2026,7,malware_download,4580
87,2026,8,malware_download,15663
88,2026,9,malware_download,15203
89,2026,10,malware_download,29380


## 2. Publicar en PostgreSQL

Escribimos la gold en la base **`warehouse`** de PostgreSQL (esquema propio del grupo, `grupo07`), que es la que consulta Metabase. Estrategia: **reemplazo total en cada corrida**.

**¿Por qué no `to_sql(..., if_exists='replace')`?** Porque `replace` hace `DROP TABLE` + `CREATE TABLE`:

* se pierde la definición que creó el DAG (clave primaria, `check` de mes, columna `actualizado_en`, comentarios);
* **falla** si existe una vista que depende de la tabla (la de la sección 3);
* Metabase pierde la descripción de las columnas.

Hacemos el mismo reemplazo total pero con `TRUNCATE` + `INSERT` **en una sola transacción** (si algo falla, queda la versión anterior), exactamente como `publicar_gold` del DAG.

**Ojo con pisar datos nuevos:** Airflow corre todos los días, así que PostgreSQL puede tener datos más recientes que tu DuckDB local. Si es así, la celda no publica (salvo `FORZAR = True`).

In [3]:
import pandas as pd
from sqlalchemy import create_engine, text

FORZAR = False   # True = publicar aunque PostgreSQL tenga más amenazas que la DuckDB local

engine = create_engine(pg_url())

# Mismo criterio que el DAG: PG_SCHEMA si está definido; si no, el primer esquema donde
# podemos crear tablas, priorizando el que se llama como el usuario (grupo07).
SQL_ESQUEMA = r"""
    select nspname from pg_namespace
    where has_schema_privilege(nspname, 'CREATE')
      and nspname not like 'pg\_%' and nspname <> 'information_schema'
    order by (nspname = current_user) desc, (nspname = 'public') desc, nspname
    limit 1
"""
with engine.connect() as c:
    usuario, base = c.execute(text("select current_user, current_database()")).one()
    esquema = os.environ.get("PG_SCHEMA") or c.execute(text(SQL_ESQUEMA)).scalar()
if not esquema:
    raise RuntimeError("El usuario no tiene permiso CREATE en ningún esquema de PostgreSQL.")
TABLA = f'"{esquema}".fct_amenazas_mensual'
print(f"Conectado a '{base}' como '{usuario}' → {TABLA}")

DDL = f"""
    create table if not exists {TABLA} (
        anio              integer     not null,
        mes               integer     not null check (mes between 1 and 12),
        tipo_amenaza      text        not null,
        cantidad_amenazas bigint      not null,
        actualizado_en    timestamptz not null default now(),
        primary key (anio, mes, tipo_amenaza)
    )
"""

with engine.connect() as c:
    existe = c.execute(text("select to_regclass(:t)"), {"t": TABLA}).scalar() is not None
    filas_pg, total_pg = (c.execute(text(f"select count(*), coalesce(sum(cantidad_amenazas), 0) from {TABLA}")).one()
                          if existe else (0, 0))
total_local = int(gold.cantidad_amenazas.sum())
print(f"PostgreSQL hoy : {filas_pg} filas, {total_pg} amenazas")
print(f"DuckDB local   : {len(gold)} filas, {total_local} amenazas")

if total_pg > total_local and not FORZAR:
    print("⚠️  PostgreSQL ya tiene datos más nuevos (los publicó el DAG): no los piso.\n"
          "    Para publicar desde acá, actualizá la DuckDB con NB04 o poné FORZAR = True.")
else:
    with engine.begin() as c:                       # una transacción: todo o nada
        c.execute(text(DDL))
        c.execute(text(f"truncate {TABLA}"))
        gold.to_sql("fct_amenazas_mensual", c, schema=esquema, if_exists="append",
                    index=False, method="multi", chunksize=1000)
        n, total = c.execute(text(f"select count(*), sum(cantidad_amenazas) from {TABLA}")).one()
    assert n == len(gold), f"Se esperaban {len(gold)} filas y hay {n}"
    print(f"✅ Publicadas {n} filas ({total} amenazas) en {TABLA}")

Conectado a 'warehouse' como 'grupo07' → "grupo07".fct_amenazas_mensual
PostgreSQL hoy : 90 filas, 77561 amenazas
DuckDB local   : 90 filas, 75519 amenazas
⚠️  PostgreSQL ya tiene datos más nuevos (los publicó el DAG): no los piso.
    Para publicar desde acá, actualizá la DuckDB con NB04 o poné FORZAR = True.


## 3. Features reproducibles + governance en PostgreSQL

La gold tiene `anio` y `mes` por separado y solo el conteo. Para el tablero (y para la E4) agregamos una **vista** con features derivadas:

| feature | qué es |
|---|---|
| `fecha` | primer día del mes (Metabase necesita una fecha para series de tiempo) |
| `total_mes` | amenazas de todos los tipos en ese mes |
| `pct_del_mes` | % que representa el tipo dentro del mes |
| `cantidad_mes_anterior` | amenazas del mismo tipo el mes calendario anterior (0 si no hubo) |
| `variacion_mensual_pct` | variación % contra el mes anterior |
| `acumulado_tipo` | suma acumulada histórica del tipo |
| `promedio_movil_3m` | promedio de los últimos 3 meses calendario (con datos) |

**Por qué es reproducible:** es código (se recrea con `drop view` + `create view`), no una copia de datos. Se recalcula sola cada vez que el DAG refresca la tabla (el `TRUNCATE` del DAG no rompe la vista) y se puede recrear corriendo esta celda las veces que haga falta.

Además dejamos **comentarios** en la tabla, la vista y cada columna: son la documentación del dataset dentro del warehouse y Metabase los muestra como descripción al sincronizar.

In [4]:
VISTA = f'"{esquema}".v_amenazas_features'

SQL_VISTA = f"""
create view {VISTA} as
with base as (
    select make_date(anio, mes, 1) as fecha, anio, mes, tipo_amenaza, cantidad_amenazas
    from {TABLA}
)
select
    b.fecha, b.anio, b.mes, b.tipo_amenaza, b.cantidad_amenazas,
    cast(sum(b.cantidad_amenazas) over (partition by b.fecha) as bigint)       as total_mes,
    round(100.0 * b.cantidad_amenazas
          / sum(b.cantidad_amenazas) over (partition by b.fecha), 2)           as pct_del_mes,
    coalesce(p.cantidad_amenazas, 0)                                            as cantidad_mes_anterior,
    case when p.cantidad_amenazas > 0
         then round(100.0 * (b.cantidad_amenazas - p.cantidad_amenazas) / p.cantidad_amenazas, 2)
    end                                                                         as variacion_mensual_pct,
    cast(sum(b.cantidad_amenazas) over (partition by b.tipo_amenaza order by b.fecha) as bigint) as acumulado_tipo,
    round(avg(b.cantidad_amenazas) over (
          partition by b.tipo_amenaza order by b.fecha
          range between interval '2 months' preceding and current row), 2)      as promedio_movil_3m
from base b
left join base p
       on p.tipo_amenaza = b.tipo_amenaza
      and p.fecha = (b.fecha - interval '1 month')::date
"""

COMENTARIOS = {
    ("table", TABLA): "Gold URLhaus (grupo07): amenazas por mes y tipo. Fuente: API URLhaus de abuse.ch. "
                      "Linaje: bronze Delta (MinIO) -> dbt stg_urlhaus -> fct_amenazas_mensual. "
                      "Refresco diario por el DAG pipeline_grupo07 (Airflow).",
    ("view", VISTA): "Features derivadas de fct_amenazas_mensual para el tablero de Metabase (grupo07).",
}
COLUMNAS = {
    "anio": "Año en que URLhaus registró la URL (date_added).",
    "mes": "Mes (1-12) en que URLhaus registró la URL.",
    "tipo_amenaza": "Tipo de amenaza informado por URLhaus (campo threat).",
    "cantidad_amenazas": "Cantidad de URLs maliciosas distintas (por id) registradas en el mes.",
    "actualizado_en": "Momento en que el DAG publicó la fila.",
    "fecha": "Primer día del mes; usar para series de tiempo.",
    "total_mes": "Total de amenazas del mes, todos los tipos.",
    "pct_del_mes": "Porcentaje del tipo sobre el total del mes.",
    "cantidad_mes_anterior": "Amenazas del mismo tipo el mes calendario anterior (0 si no hubo).",
    "variacion_mensual_pct": "Variación % contra el mes anterior (vacío si el mes anterior fue 0).",
    "acumulado_tipo": "Suma acumulada histórica del tipo hasta el mes.",
    "promedio_movil_3m": "Promedio de cantidad_amenazas en los últimos 3 meses calendario con datos.",
}

def literal(s):   # COMMENT ON no acepta parámetros: escapamos las comillas a mano
    return "'" + s.replace("'", "''") + "'"

with engine.begin() as c:
    # drop + create (y no solo "create or replace") para poder cambiar tipos o columnas
    # de la vista más adelante; nada en la base depende de ella, Metabase la busca por nombre.
    c.execute(text(f"drop view if exists {VISTA}"))
    c.execute(text(SQL_VISTA))
    for (tipo, obj), txt in COMENTARIOS.items():
        c.execute(text(f"comment on {tipo} {obj} is {literal(txt)}"))
    for obj in (TABLA, VISTA):
        cols = c.execute(text("select attname from pg_attribute where attrelid = to_regclass(:o) "
                              "and attnum > 0 and not attisdropped"), {"o": obj}).scalars().all()
        for col in cols:
            if col in COLUMNAS:
                c.execute(text(f'comment on column {obj}."{col}" is {literal(COLUMNAS[col])}'))

features = pd.read_sql(text(f"select * from {VISTA} order by fecha desc, cantidad_amenazas desc"), engine)
print(f"✅ Vista {VISTA}: {len(features)} filas")
features.head(12)

✅ Vista "grupo07".v_amenazas_features: 90 filas


,fecha,anio,mes,tipo_amenaza,cantidad_amenazas,total_mes,pct_del_mes,cantidad_mes_anterior,variacion_mensual_pct,acumulado_tipo,promedio_movil_3m
0,2026-10-01,2026,10,malware_download,31422,31422,100.0,15203,106.68,77561,20762.67
1,2026-09-01,2026,9,malware_download,15203,15203,100.0,15663,-2.94,46139,11815.33
2,2026-08-01,2026,8,malware_download,15663,15663,100.0,4580,241.99,30936,8325.67
3,2026-07-01,2026,7,malware_download,4580,4580,100.0,4734,-3.25,15273,3128.67
4,2026-06-01,2026,6,malware_download,4734,4734,100.0,72,6475.00,10693,1706.33
5,2026-05-01,2026,5,malware_download,72,72,100.0,313,-77.00,5959,209.67
6,2026-04-01,2026,4,malware_download,313,313,100.0,244,28.28,5887,232.33
7,2026-03-01,2026,3,malware_download,244,244,100.0,140,74.29,5574,168.33
8,2026-02-01,2026,2,malware_download,140,140,100.0,121,15.70,5330,111.67
9,2026-01-01,2026,1,malware_download,121,121,100.0,74,63.51,5190,111.33


## 4. Verificación

Reconciliación entre capas: lo que está en PostgreSQL (tabla y vista) tiene que coincidir con la gold. Si el DAG publicó datos más nuevos que tu DuckDB local, la tabla puede tener más; lo importante es que tabla y vista coincidan entre sí.

In [5]:
with engine.connect() as c:
    t = c.execute(text(f"select count(*), sum(cantidad_amenazas), min(make_date(anio, mes, 1)), "
                       f"max(make_date(anio, mes, 1)), max(actualizado_en) from {TABLA}")).one()
    v = c.execute(text(f"select count(*), sum(cantidad_amenazas) from {VISTA}")).one()
print(f"Tabla : {t[0]} filas, {t[1]} amenazas, de {t[2]} a {t[3]}, publicada {t[4]}")
print(f"Vista : {v[0]} filas, {v[1]} amenazas")
print(f"DuckDB: {len(gold)} filas, {total_local} amenazas")
assert (t[0], t[1]) == (v[0], v[1]), "La vista no coincide con la tabla"
print("✅ Tabla y vista coinciden" + (" con la DuckDB local" if t[1] == total_local else
                                     " (PostgreSQL tiene datos más nuevos que la DuckDB local)"))

Tabla : 90 filas, 77561 amenazas, de 2018-06-01 a 2026-10-01, publicada 2026-10-26 03:00:25.403333+00:00
Vista : 90 filas, 77561 amenazas
DuckDB: 90 filas, 75519 amenazas
✅ Tabla y vista coinciden (PostgreSQL tiene datos más nuevos que la DuckDB local)


In [6]:
# Datos de conexión para Metabase (sin contraseña: esa se escribe solo en Metabase)
print("Tipo      : PostgreSQL")
print("Host      :", os.environ.get("PGHOST", "postgres"))
print("Puerto    :", os.environ.get("PGPORT", "5432"))
print("Base      :", base)
print("Usuario   :", usuario)
print("Esquema   :", esquema, "→ tablas: fct_amenazas_mensual, v_amenazas_features")

Tipo      : PostgreSQL
Host      : postgres
Puerto    : 5432
Base      : warehouse
Usuario   : grupo07
Esquema   : grupo07 → tablas: fct_amenazas_mensual, v_amenazas_features


## 5. Construir el tablero en Metabase

Esta parte es manual, en **https://metabase.nexoacademia.org**.

**Paso 0 · ¿Está conectada la base?** Entrá a *Explorar datos* (Browse data) y buscá `warehouse` → esquema `grupo07`.
* Si aparece: seguí al paso 1. Si no ves `v_amenazas_features`, pedí (o hacé, si sos admin) *Admin → Bases de datos → warehouse → Sincronizar esquema ahora*.
* Si no aparece y **sos admin**: *Admin → Bases de datos → Agregar base de datos* → PostgreSQL, con los datos que imprimió la celda anterior. La contraseña es la de `PGPASSWORD` del JupyterHub: escribila solo en Metabase, nunca en el notebook.
* Si no aparece y **no sos admin**: pedile al profesor que conecte `warehouse` (o que te dé permisos sobre el esquema `grupo07`).

**Paso 1 · Preguntas** (*Nuevo → Pregunta*, todas sobre `V Amenazas Features` salvo que se diga otra cosa):

| # | Nombre | Cómo | Visualización |
|---|---|---|---|
| 1 | Total de amenazas | Resumir: *Suma de Cantidad Amenazas* | Número |
| 2 | Amenazas por mes | Suma de Cantidad Amenazas, agrupar por *Fecha: Mes* | Línea |
| 3 | Amenazas por tipo | Suma de Cantidad Amenazas, agrupar por *Tipo Amenaza*, ordenar desc | Barras |
| 4 | Tendencia (promedio móvil 3 meses) | Suma de Cantidad Amenazas y Suma de Promedio Movil 3M por *Fecha: Mes* | Línea (dos series) |
| 5 | Últimos 12 meses | Filtro *Fecha: últimos 12 meses*; columnas fecha, tipo, cantidad, pct_del_mes, variacion_mensual_pct | Tabla |

Si preferís SQL (*Nuevo → Consulta SQL*), por ejemplo para la 2:

```sql
select fecha, sum(cantidad_amenazas) as amenazas
from grupo07.v_amenazas_features
group by fecha
order by fecha
```

**Paso 2 · Dashboard** (*Nuevo → Dashboard*, nombre sugerido: **"Amenazas URLhaus — grupo07"**):
1. Arriba: la tarjeta *Total de amenazas* y una tarjeta de texto con qué es el dataset, la fuente (URLhaus, abuse.ch) y que se actualiza a diario.
2. Al medio: *Amenazas por mes* y *Tendencia* a lo ancho.
3. Abajo: *Amenazas por tipo* y *Últimos 12 meses*.
4. Agregá un filtro de **Fecha** (conectado a `fecha` en cada tarjeta) y uno de **Tipo de amenaza** (`tipo_amenaza`). Guardá.

> **Metabase vs. Grafana:** Metabase responde preguntas de *negocio* sobre el warehouse (esta clase); Grafana se usa para *observabilidad* del pipeline (semana 13). Cada herramienta en su rol.

> **Para leer bien los datos:** `fecha` es cuándo URLhaus *registró* la URL, no cuándo ocurrió el ataque; el mes en curso está incompleto; y en URLhaus casi todo viene como un único tipo (`malware_download`), así que la pregunta 3 va a mostrar pocas barras.

## 6. Governance del dataset (ficha para la E4)

| | |
|---|---|
| **Dataset** | `warehouse.grupo07.fct_amenazas_mensual` (gold) y `warehouse.grupo07.v_amenazas_features` (features) |
| **Dueño** | grupo07 — Ingeniería de Datos, LCD UCA Rosario 2026 |
| **Propósito** | Seguir la evolución mensual de URLs maliciosas reportadas, por tipo de amenaza |
| **Fuente** | API de URLhaus (abuse.ch). Requiere API key propia; uso sujeto a los términos de abuse.ch |
| **Linaje** | API → `extraer_bronze` (Delta en `s3://grupo07/bronze/series`, incremental por id) → dbt `stg_urlhaus` (silver: tipado y deduplicado por id) → dbt `fct_amenazas_mensual` (gold, DuckDB) → `publicar_gold` (PostgreSQL) → vista de features → Metabase |
| **Grano** | una fila por (`anio`, `mes`, `tipo_amenaza`) — clave primaria en PostgreSQL |
| **Frecuencia** | diaria (`@daily`), DAG `pipeline_grupo07` en Airflow; `actualizado_en` indica la última publicación |
| **Calidad** | tests dbt: `id_amenaza` not_null + unique, `fecha_agregado` not_null, `anio`/`tipo_amenaza` not_null, `mes` en 1..12; PK y `check` en PostgreSQL; el DAG no publica una gold vacía y verifica el conteo; reconciliación tabla = vista (sección 4) |
| **Seguridad y acceso** | credenciales solo como variables de entorno y en `dags/.env` (no versionado; la API key se pide con `getpass`); Metabase lee con el usuario del grupo |
| **Datos sensibles** | la gold es agregada: no incluye URLs ni el usuario que reportó (minimización). Las URLs maliciosas quedan solo en bronze/silver |
| **Limitaciones** | la fecha es la de registro en URLhaus; el mes en curso está incompleto; la API solo devuelve lo reciente, el histórico depende de lo ya acumulado en bronze |
| **Diccionario** | cada columna tiene su descripción como `comment` en PostgreSQL (sección 3), visible en Metabase |

## Entrega 4 (E4)

Lo que se evalúa: capa gold en PostgreSQL, features reproducibles, tablero de negocio en Metabase y documentación de governance del dataset. Enunciado y rúbrica en la semana 13.

| Requisito | Dónde queda resuelto |
|---|---|
| Capa gold en PostgreSQL | DAG `publicar_gold` (diario) + sección 2 de este notebook |
| Features reproducibles | vista `grupo07.v_amenazas_features` (sección 3), recreable con una celda |
| Tablero de negocio | Metabase, sección 5 (captura del dashboard para la entrega) |
| Governance | ficha de la sección 6 + comentarios en PostgreSQL |